# RUNX motif accessibility in dSSc skin fibroblasts — PREP

Implements the setup for `RR_PREREG_runx2_motif_accessibility_2026-09-09.md`.

**This notebook does preparation only. It does not run the test.**
The test runs as a script in a separate process (last cell), so the preregistered
controls cannot be overridden by anything left in this kernel.

**Design rules, taken from the P34 post mortem this week:**

1. Every cell that must gate **raises**. It does not print a warning and continue.
   The P34 pipeline's own inspection cell crashed at `0/16` markers and the run
   carried on regardless.
2. Every value a human must supply starts as a sentinel that raises if untouched.
   P34 shipped `TOP_CLUSTERS = ['2', '3']   # <-- REPLACE with actual cluster IDs`
   into an executed run.
3. Cells assert their own preconditions, so out-of-order execution fails loudly
   rather than silently using stale state.
4. The run writes a timestamped log to Drive. Both pipeline notebooks in the
   GitHub repos are committed with empty outputs; a log is the only durable
   evidence a run happened.

## 1. Paths and Drive

In [ ]:
from google.colab import drive
import os, datetime, sys

drive.mount('/content/drive')

WORK  = '/content/drive/MyDrive/RR/runx_motif'
CACHE = '/content/drive/MyDrive/RR/_cache'          # genome + motifs, survives sessions
os.makedirs(WORK, exist_ok=True)
os.makedirs(CACHE, exist_ok=True)
RUN_ID = datetime.datetime.now().strftime('%Y%m%d_%H%M')
print('WORK  :', WORK)
print('CACHE :', CACHE)
print('RUN_ID:', RUN_ID)

## 2. Install

In [ ]:
!pip -q install anndata scanpy muon pyjaspar pyfaidx MOODS-python 2>&1 | tail -2
import anndata as ad, scanpy as sc, numpy as np, pandas as pd, scipy.sparse as sp
print('anndata', ad.__version__, '| scanpy', sc.__version__)

## 3. ⛔ GATE ZERO — is the ATAC actually deposited?

The Gur paper's data availability statement names only scRNA-seq. The multiome
ATAC may not be under GSE195452 at all. **Nothing below this cell is worth
setting up until this is answered.**

This cell raises if it cannot find evidence of ATAC/multiome files.

In [ ]:
import re, urllib.request

GSE = 'GSE195452'
stem = GSE[:-3] + 'nnn'
ftp  = f'https://ftp.ncbi.nlm.nih.gov/geo/series/{stem}/{GSE}/suppl/'

print('Supplementary files deposited under', GSE)
print('-' * 70)
try:
    html = urllib.request.urlopen(ftp, timeout=60).read().decode('utf8', 'ignore')
    files = sorted(set(re.findall(r'href="([^"?][^"]*)"', html)))
    files = [f for f in files if not f.startswith('/')]
    for f in files:
        print('  ', f)
except Exception as e:
    files = []
    print('  could not list FTP:', e)

blob = ' '.join(files).lower()
ATAC_HINTS = ['atac', 'multiome', 'peak', 'fragment', 'tsv.gz']
found = [h for h in ATAC_HINTS if h in blob]

print('-' * 70)
print('ATAC-like hints found:', found or 'NONE')

if not found:
    raise RuntimeError(
        'GATE ZERO FAILED: no ATAC/multiome/peak/fragment files visible under '
        f'{GSE}. The chromatin data may not be deposited.\n'
        'Do NOT proceed here. Options:\n'
        '  (a) check the GSM sample list in a browser for multiome samples\n'
        '  (b) fall back to GSE312129 / GSE312932 (snRNA + snATAC, not paired\n'
        '      in the same cells, which costs the paralog attribution step)\n'
        '  (c) email the authors — noting this program has recorded that cold\n'
        '      data requests to corresponding authors have never been answered.')
print('\nGATE ZERO PASSED. Continue.')

## 4. Reference cache — hg38 and JASPAR

~3 GB. Cached to Drive so it downloads once, not once per session.

In [ ]:
FA = f'{CACHE}/hg38.fa'
if not os.path.exists(FA):
    print('downloading hg38 (one time, several minutes)...')
    !wget -q -O {CACHE}/hg38.fa.gz https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz
    !gunzip -f {CACHE}/hg38.fa.gz
else:
    print('hg38 already cached')

from pyfaidx import Fasta
genome = Fasta(FA)
print('contigs:', len(genome.keys()))

from pyjaspar import jaspardb
jdb = jaspardb(release='JASPAR2024')
motifs = jdb.fetch_motifs(collection='CORE', tax_group=['vertebrates'])
print('JASPAR CORE vertebrate motifs:', len(motifs))

## 5. Load the objects, and print what is actually in them

**Inspection only.** Nothing is chosen here. Read the printed keys, then set the
values in cell 6.

In [ ]:
# EDIT: point these at your downloaded files
ATAC_PATH = f'{WORK}/atac_peaks.h5ad'
RNA_PATH  = f'{WORK}/rna.h5ad'

for p in (ATAC_PATH, RNA_PATH):
    if not os.path.exists(p):
        raise FileNotFoundError(f'missing: {p}\nDownload the objects into {WORK} first.')

atac = ad.read_h5ad(ATAC_PATH)
rna  = ad.read_h5ad(RNA_PATH)
print('ATAC', atac.shape)
print('RNA ', rna.shape)
print('\nATAC .obs columns:'); print(' ', list(atac.obs.columns))
print('\nATAC .var columns:'); print(' ', list(atac.var.columns))
print('\nfirst 3 peak names:', list(atac.var_names[:3]))
print('\ncandidate donor columns (many unique values):')
for c in atac.obs.columns:
    n = atac.obs[c].nunique()
    if 2 <= n <= 40:
        print(f'  {c:<28} {n:>3} unique   {list(atac.obs[c].unique()[:6])}')

## 6. ⛔ HARD GATE — you must set these, and wrong values raise

Sentinels. If you run this cell without editing, it stops.
This is the cell that P34 did not have.

In [ ]:
# EDIT BOTH. Leaving the sentinel raises.
DONOR_COL = '<<SET ME>>'      # e.g. 'donor', 'Sample', 'patient'
GROUP_COL = '<<SET ME>>'      # column separating healthy from dSSc

# Map whatever the file calls the two groups onto exactly 'HC' and 'dSSc'.
GROUP_MAP = {
    # '<value in file>': 'HC',
    # '<value in file>': 'dSSc',
}

if '<<SET ME>>' in (DONOR_COL, GROUP_COL):
    raise RuntimeError('STOP: set DONOR_COL and GROUP_COL from the printout in cell 5.')
for c in (DONOR_COL, GROUP_COL):
    if c not in atac.obs:
        raise RuntimeError(f'STOP: {c!r} is not an .obs column. Options: {list(atac.obs.columns)}')
if not GROUP_MAP:
    raise RuntimeError('STOP: GROUP_MAP is empty. Map the file values onto HC and dSSc.')

atac.obs['donor'] = atac.obs[DONOR_COL].astype(str)
atac.obs['group'] = atac.obs[GROUP_COL].astype(str).map(GROUP_MAP)
if atac.obs['group'].isna().any():
    bad = atac.obs.loc[atac.obs['group'].isna(), GROUP_COL].unique()[:10]
    raise RuntimeError(f'STOP: GROUP_MAP does not cover these values: {list(bad)}')

tab = atac.obs.groupby(['group', 'donor'], observed=True).size()
n_hc = atac.obs.loc[atac.obs.group == 'HC', 'donor'].nunique()
n_ss = atac.obs.loc[atac.obs.group == 'dSSc', 'donor'].nunique()
print(f'HC donors: {n_hc}   dSSc donors: {n_ss}')
print(tab.to_string())

if min(n_hc, n_ss) < 3:
    raise RuntimeError(f'STOP: fewer than 3 donors in a group (HC={n_hc}, dSSc={n_ss}). '
                       'A donor-level test is not defensible.')
print('\nGATE PASSED.')

## 7. Build GC content and the peaks-by-motifs matrix

This is the real setup work. It is also the step that decides what the deviation
statistic is comparing against, so the background matching in the script depends
on `var['gc']` being right.

In [ ]:
assert 'group' in atac.obs, 'run cell 6 first'

# peak names -> chrom, start, end. Handles 'chr1:100-200' and 'chr1-100-200'.
def parse_peaks(names):
    rows = []
    for p in names:
        s = str(p).replace(':', '-')
        parts = s.rsplit('-', 2)
        if len(parts) != 3:
            raise RuntimeError(f'cannot parse peak name {p!r}. Expected chr:start-end.')
        rows.append((parts[0], int(parts[1]), int(parts[2])))
    return pd.DataFrame(rows, columns=['chrom', 'start', 'end'])

peaks = parse_peaks(atac.var_names)
print(peaks.head())

seqs, gc = [], []
for c, s, e in peaks.itertuples(index=False):
    try:
        seq = str(genome[c][s:e]).upper()
    except Exception:
        seq = ''
    seqs.append(seq)
    gc.append((seq.count('G') + seq.count('C')) / len(seq) if seq else 0.5)

atac.var['gc'] = gc
print(f'GC computed for {len(gc):,} peaks. median {np.median(gc):.3f}')
n_empty = sum(1 for s in seqs if not s)
if n_empty:
    print(f'WARNING: {n_empty:,} peaks had no sequence (contig naming mismatch?)')
    if n_empty > 0.05 * len(seqs):
        raise RuntimeError('STOP: >5% of peaks got no sequence. Check chrom naming '
                           '(chr1 vs 1) against the genome contigs.')

In [ ]:
import MOODS.scan, MOODS.tools

KEEP = ['RUNX1','RUNX2','RUNX3','SOX9','SMAD3','TWIST1','FOSL2','JUNB','TEAD1',
        'HNF4A','POU5F1','RUNX1::CBFB']
sel = [m for m in motifs if m.name.upper() in [k.upper() for k in KEEP]]
print('motifs selected:', [m.name for m in sel])
missing = set(k.upper() for k in KEEP) - set(m.name.upper() for m in sel)
if missing:
    print('NOT FOUND in JASPAR CORE:', missing)

matrices, thresholds, names = [], [], []
for m in sel:
    counts = [list(m.counts[b]) for b in 'ACGT']
    pwm = MOODS.tools.log_odds(counts, [0.25]*4, 1)
    matrices.append(pwm)
    thresholds.append(MOODS.tools.threshold_from_p(pwm, [0.25]*4, 1e-4))
    names.append(m.name)

M = np.zeros((atac.n_vars, len(names)), dtype=np.int8)
for i, seq in enumerate(seqs):
    if not seq: continue
    hits = MOODS.scan.scan_dna(seq, matrices, [0.25]*4, thresholds)
    for j, h in enumerate(hits):
        if len(h): M[i, j] = 1
    if i % 20000 == 0: print(f'  scanned {i:,}/{len(seqs):,}')

atac.varm['motif_match'] = M
atac.uns['motif_names'] = names
print('\npeaks per motif:')
for j, n in enumerate(names):
    print(f'  {n:<14} {int(M[:, j].sum()):>7,}')

## 8. Save the prepared objects

In [ ]:
ATAC_OUT = f'{WORK}/atac_prepped.h5ad'
RNA_OUT  = f'{WORK}/rna_prepped.h5ad'
rna.obs['donor'] = atac.obs['donor'].reindex(rna.obs_names).values
rna.obs['group'] = atac.obs['group'].reindex(rna.obs_names).values
atac.write_h5ad(ATAC_OUT); rna.write_h5ad(RNA_OUT)
print('wrote', ATAC_OUT); print('wrote', RNA_OUT)

## 9. Write the test script to Drive

The script is authored here so there is one file to upload, not two. It is written
to Drive verbatim and then hashed, so the log identifies exactly which version ran.

**Edit `POS_CONTROL` in this cell before you run it.** The preregistration says the
positive controls come from Gur's 63 enriched motifs. The placeholder below is mine,
not theirs. That choice has to be locked before you look at RUNX, or the controls
stop being controls. Whatever you set here is what the hash records.


In [ ]:
SCRIPT = f'{WORK}/runx_motif_test.py'

SCRIPT_SRC = r'''
#!/usr/bin/env python3
"""
runx_motif_test.py  -  is the RUNX regulon engaged in dSSc skin fibroblasts?

Implements RR_PREREG_runx2_motif_accessibility_2026-09-09.md. Every choice in
that file is fixed; nothing here is tunable after seeing results.

WHAT THIS DOES DIFFERENTLY FROM A NAIVE chromVAR RUN, AND WHY EACH MATTERS

  1. DONOR-LEVEL TESTING. Per-cell deviations are aggregated to a donor mean and
     tested at n=9 vs 9. Cells from one donor are not independent. The cell-level
     version of this mistake returned >50% of the transcriptome as "significant"
     in P45 this week; the same error in ATAC would be harder to spot.

  2. THE RUNX PARALOG PROBLEM IS TREATED AS THE MAIN THREAT, NOT A FOOTNOTE.
     RUNX1/2/3 share a near-identical core motif (TGTGGT). Accessibility alone
     CANNOT attribute a deviation to RUNX2. This script therefore refuses to
     report a RUNX result without the paired-RNA attribution in step 5, which is
     only possible because the data is multiome.

  3. CONTROLS GATE THE RESULT. If the preregistered positive controls do not
     move, the script says the pipeline failed rather than that the hypothesis
     failed. Those are different findings and conflating them is how a null
     becomes a false claim.

  4. THE COLLAGEN CONFOUND IS CHECKED. In this same cohort the osteogenic RNA
     score was 91% collagen and REVERSED SIGN once fibrosis was regressed out.
     Any motif result is reported raw AND residual, and a sign flip is reported
     in the same breath as the raw number.

USAGE
    python runx_motif_test.py --h5mu path.h5mu
    python runx_motif_test.py --atac atac.h5ad --rna rna.h5ad
"""
import argparse, sys
import numpy as np, pandas as pd
from scipy import stats
import scipy.sparse as sp

# ---- preregistered, do not edit after seeing data -------------------------
PRIMARY      = ["RUNX1", "RUNX2", "RUNX3"]     # scored together; see paralog note
SECONDARY    = ["SOX9"]
NEG_CONTROL  = ["HNF4A", "POU5F1"]             # lineage-inappropriate
POS_CONTROL  = ["SMAD3", "TWIST1", "RUNX1",    # pick from Gur's 63 BEFORE running
                "FOSL2", "JUNB", "TEAD1"]      # <- RECORD which were used
MYOFIB_GENES = ["ACTA2", "POSTN", "COL1A1", "COL1A2", "TAGLN", "CCN2"]
DONOR_KEY, GROUP_KEY = "donor", "group"


def deviations(atac, motif_key="motif_match", n_bg=50, seed=0):
    """
    chromVAR-style deviation, implemented transparently so the statistic is
    auditable rather than a black box.

    For motif m: observed accessibility of peaks carrying m, minus the mean of
    n_bg background peak sets matched on GC content and mean accessibility,
    divided by the background SD. The matching is what makes the statistic
    comparable across motifs of different peak counts.
    """
    rng = np.random.default_rng(seed)
    X = atac.X.tocsr() if sp.issparse(atac.X) else sp.csr_matrix(atac.X)
    peak_acc = np.asarray(X.sum(0)).ravel()
    gc = atac.var["gc"].values if "gc" in atac.var else np.full(atac.n_vars, 0.5)

    # bin peaks on (GC, accessibility) so backgrounds are matched, not random
    def qbin(v, k=20):
        r = pd.qcut(pd.Series(v).rank(method="first"), k, labels=False)
        return r.values
    strata = qbin(gc) * 100 + qbin(np.log1p(peak_acc))
    by_stratum = {s: np.where(strata == s)[0] for s in np.unique(strata)}

    M = atac.varm[motif_key]                      # peaks x motifs, binary
    names = list(atac.uns.get("motif_names", range(M.shape[1])))
    out = np.zeros((atac.n_obs, M.shape[1]), dtype=np.float32)

    cell_tot = np.asarray(X.sum(1)).ravel(); cell_tot[cell_tot == 0] = 1
    for j in range(M.shape[1]):
        idx = np.flatnonzero(np.asarray(M[:, j]).ravel())
        if len(idx) < 10:
            out[:, j] = np.nan; continue
        obs = np.asarray(X[:, idx].sum(1)).ravel() / cell_tot
        bg = np.empty((n_bg, atac.n_obs), dtype=np.float32)
        for b in range(n_bg):
            pick = [rng.choice(by_stratum[strata[i]]) for i in idx]
            bg[b] = np.asarray(X[:, pick].sum(1)).ravel() / cell_tot
        mu, sd = bg.mean(0), bg.std(0); sd[sd == 0] = 1
        out[:, j] = (obs - mu) / sd
    return pd.DataFrame(out, index=atac.obs_names, columns=names)


def donor_test(dev, obs, motifs):
    """Aggregate to donor, then Mann-Whitney at n=9 vs 9. Never cell-level."""
    d = dev.copy()
    d[DONOR_KEY] = obs[DONOR_KEY].values
    d[GROUP_KEY] = obs[GROUP_KEY].values
    per_donor = d.groupby([DONOR_KEY, GROUP_KEY], observed=True).mean(numeric_only=True).reset_index()
    rows = []
    for m in motifs:
        if m not in per_donor: continue
        a = per_donor.loc[per_donor[GROUP_KEY] == "HC", m].dropna()
        b = per_donor.loc[per_donor[GROUP_KEY] == "dSSc", m].dropna()
        if len(a) < 3 or len(b) < 3: continue
        rows.append(dict(motif=m, n_HC=len(a), n_dSSc=len(b),
                         mean_HC=a.mean(), mean_dSSc=b.mean(),
                         delta=b.mean() - a.mean(),
                         direction="dSSc UP" if b.mean() > a.mean() else "dSSc DOWN",
                         p=stats.mannwhitneyu(b, a, alternative="two-sided").pvalue))
    r = pd.DataFrame(rows)
    if len(r):
        p = r.p.values; o = np.argsort(p); m_ = len(p); q = np.empty(m_); run = 1.0
        for i in range(m_ - 1, -1, -1):
            run = min(run, p[o[i]] * m_ / (i + 1)); q[o[i]] = run
        r["q_BH"] = q; r["survives_q05"] = r.q_BH < 0.05
    return r, per_donor


def attribute_runx(dev, rna, obs):
    """
    STEP 5, AND THE RESULT IS NOT REPORTABLE WITHOUT IT.
    RUNX1/2/3 share a motif. Ask which paralog's RNA tracks the deviation in the
    SAME cells. This is what multiome buys and ATAC alone cannot answer.
    """
    print("\n=== RUNX PARALOG ATTRIBUTION (the decisive step) ===")
    if "RUNX" not in " ".join(dev.columns):
        print("  no RUNX motif column"); return
    col = [c for c in dev.columns if "RUNX" in c.upper()][0]
    print(f"  motif column: {col}")
    for g in ["RUNX1", "RUNX2", "RUNX3"]:
        if g not in rna.var_names:
            print(f"  {g:<6} not in RNA matrix"); continue
        x = rna[:, g].X
        x = np.asarray(x.todense()).ravel() if sp.issparse(x) else np.asarray(x).ravel()
        rho = stats.spearmanr(dev[col].values, x).statistic
        pct = 100 * (x > 0).mean()
        print(f"  {g:<6} rho(deviation, RNA) = {rho:+.3f}   detected in {pct:5.1f}% of cells")
    print("\n  READING: if the deviation tracks RUNX1 rather than RUNX2, the signal is")
    print("  NOT osteogenic and a naive report would have been wrong. If RUNX2 is")
    print("  detected in under ~2% of cells, no attribution is possible and the")
    print("  honest output is that this dataset cannot answer the question.")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--h5mu"); ap.add_argument("--atac"); ap.add_argument("--rna")
    a = ap.parse_args()
    if not (a.h5mu or (a.atac and a.rna)):
        print(__doc__); sys.exit(0)
    import anndata as ad
    if a.h5mu:
        import muon
        mu = muon.read(a.h5mu); atac, rna = mu.mod["atac"], mu.mod["rna"]
    else:
        atac, rna = ad.read_h5ad(a.atac), ad.read_h5ad(a.rna)

    print(f"ATAC {atac.shape}   RNA {rna.shape}")
    print(f"donors: {atac.obs[DONOR_KEY].nunique()}   groups: {dict(atac.obs[GROUP_KEY].value_counts())}")

    dev = deviations(atac)

    print("\n=== CONTROLS FIRST. THE RESULT IS GATED ON THESE. ===")
    ctrl, _ = donor_test(dev, atac.obs, POS_CONTROL + NEG_CONTROL)
    print(ctrl.to_string(index=False))
    pos_ok = ctrl[ctrl.motif.isin(POS_CONTROL)].survives_q05.any() if len(ctrl) else False
    neg_bad = ctrl[ctrl.motif.isin(NEG_CONTROL)].survives_q05.any() if len(ctrl) else False
    if not pos_ok:
        print("\n  STOP. No preregistered positive control moved. The finding is that the")
        print("  PIPELINE did not work, not that the hypothesis failed. Do not report a null.")
        return
    if neg_bad:
        print("\n  STOP. A negative control moved. The deviation background is mis-modeled.")
        return
    print("\n  Controls pass. Proceeding.")

    print("\n=== PRIMARY AND SECONDARY ===")
    res, per_donor = donor_test(dev, atac.obs, PRIMARY + SECONDARY)
    print(res.to_string(index=False))

    print("\n=== COLLAGEN CONFOUND CHECK (this cohort reversed sign on the RNA score) ===")
    g = [x for x in MYOFIB_GENES if x in rna.var_names]
    if g:
        Xm = rna[:, g].X
        Xm = np.asarray(Xm.todense()) if sp.issparse(Xm) else np.asarray(Xm)
        myo = pd.Series(((Xm - Xm.mean(0)) / (Xm.std(0) + 1e-9)).mean(1), index=rna.obs_names)
        pd_myo = myo.groupby(atac.obs[DONOR_KEY].values).mean()
        for m in PRIMARY + SECONDARY:
            if m not in per_donor: continue
            sub = per_donor.dropna(subset=[m]).copy()
            sub["myo"] = pd_myo.reindex(sub[DONOR_KEY]).values
            sub = sub.dropna(subset=["myo"])
            if len(sub) < 6: continue
            b, c = np.polyfit(sub.myo, sub[m], 1)
            sub["res"] = sub[m] - (b * sub.myo + c)
            A = sub.loc[sub[GROUP_KEY] == "HC", "res"]; B = sub.loc[sub[GROUP_KEY] == "dSSc", "res"]
            if len(A) < 3 or len(B) < 3: continue
            raw_dir = res.loc[res.motif == m, "direction"].iloc[0]
            res_dir = "dSSc UP" if B.mean() > A.mean() else "dSSc DOWN"
            flag = "   *** SIGN FLIPS ***" if raw_dir != res_dir else ""
            print(f"  {m:<8} raw {raw_dir}   residual {res_dir}  "
                  f"p={stats.mannwhitneyu(B, A).pvalue:.3f}{flag}")

    attribute_runx(dev, rna, atac.obs)

    print("\n=== WHAT THIS DOES NOT LICENSE ===")
    print("  GSE195452 carries no calcinosis label. This tests the DISEASE, not the")
    print("  lesion. No statement about calcinosis follows from any result above.")


if __name__ == "__main__":
    main()
'''

with open(SCRIPT, 'w') as fh:
    fh.write(SCRIPT_SRC.lstrip('\n'))

import hashlib
h = hashlib.sha256(open(SCRIPT, 'rb').read()).hexdigest()
print('wrote', SCRIPT)
print('script sha256:', h[:16])

# echo the controls that were actually written, so the log carries them
for line in SCRIPT_SRC.splitlines():
    if line.startswith(('PRIMARY', 'SECONDARY', 'NEG_CONTROL', 'POS_CONTROL')):
        print(' ', line)


## 10. Run the test — separate process, not `%run`

`!python` spawns a fresh interpreter that sees only the file. Nothing left in this
kernel can override the preregistered controls. `%run` would inherit every variable
defined above, which is exactly what a preregistered test must not allow.

In [ ]:
LOG = f'{WORK}/run_{RUN_ID}.log'
!cd {WORK} && python runx_motif_test.py \
    --atac atac_prepped.h5ad --rna rna_prepped.h5ad 2>&1 | tee {LOG}

print('\nlog written to', LOG)

## 11. What this cannot tell you

GSE195452 carries **no calcinosis label**. Whatever comes back is a statement about
diffuse SSc skin fibroblasts, not about the lesion. It cannot adjudicate the
Bao versus Nasrallah disagreement. It can only make one side more or less plausible.

And if the positive controls did not move, the finding is that the **pipeline** did
not work — not that the hypothesis failed. Those are different results and the
script stops rather than letting them be conflated.